# Model 1 — YOLO11n Face Detector

WIDER Face YOLO data đã có split. Notebook chỉ tạo `data.yaml` rồi train. Class duy nhất: `0 = face`.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input/datasets/canomercik/wider-face-dataset-for-yolov12-format')
OUTPUT_DIR = Path('/kaggle/working/model-1-face-detector')
CONFIG_DIR = Path('/kaggle/working/face-detector-config')
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
WORKERS = 4

DATASET_ROOT = next(path for path in [INPUT_ROOT, *INPUT_ROOT.rglob('*')] if (path / 'train' / 'images').is_dir())
VAL_SPLIT = 'valid' if (DATASET_ROOT / 'valid' / 'images').is_dir() else 'val'
assert (DATASET_ROOT / VAL_SPLIT / 'images').is_dir(), 'WIDER validation split not found'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

data_yaml = CONFIG_DIR / 'face.yaml'
data_yaml.write_text(f"path: {DATASET_ROOT}\ntrain: train/images\nval: {VAL_SPLIT}/images\nnames:\n  0: face\n", encoding='utf-8')
print(data_yaml.read_text())


In [ ]:
import subprocess
import torch

subprocess.run(['pip', '-q', 'install', 'ultralytics'], check=True)
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo11n.pt')
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=device,
    project='/kaggle/working',
    name='model-1-face-detector',
    exist_ok=True,
)
print('Cloud checkpoint: /kaggle/working/model-1-face-detector/weights/best.pt')
